In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import torch

sys.path.append(str(Path("..").resolve()))   # rend le dossier projet1_avions importable
from src.dataset import LEVELS, build_hierarchy, encode_labels, load_labels

AIRCRAFT_DIR = Path("..") / "data" / "fgvc-aircraft-2013b" / "data"

In [2]:
df = load_labels(AIRCRAFT_DIR)
df, classes = encode_labels(df)
variant_to_family, family_to_manufacturer = build_hierarchy(df)

print("Classes par niveau :", {level: len(c) for level, c in classes.items()})
print("Tailles des tables :", len(variant_to_family), len(family_to_manufacturer))
df.head(3)

Classes par niveau : {'manufacturer': 30, 'family': 70, 'variant': 100}
Tailles des tables : 100 70


,image_id,manufacturer,family,variant,split,manufacturer_idx,family_idx,variant_idx
0,1025794,Boeing,Boeing 707,707-320,train,4,12,0
1,1340192,Boeing,Boeing 707,707-320,train,4,12,0
2,0056978,Boeing,Boeing 707,707-320,train,4,12,0


In [3]:
# Exemple : on remonte la hiérarchie à partir de l'indice de variante
row = df.iloc[0]
f = variant_to_family[row.variant_idx].item()
m = family_to_manufacturer[f].item()
print(f"{row.variant} (variante {row.variant_idx}) -> {classes['family'][f]} (famille {f}) -> {classes['manufacturer'][m]} (constructeur {m})")

# Vérification sur les 10 000 images d'un coup
v = torch.tensor(df["variant_idx"].values)
m_calcule = family_to_manufacturer[variant_to_family[v]]
print("Cohérence :", (m_calcule == torch.tensor(df["manufacturer_idx"].values)).all().item())

707-320 (variante 0) -> Boeing 707 (famille 12) -> Boeing (constructeur 4)
Cohérence : True


In [4]:
# Étapes 2.2 et 2.3 : transformations, Dataset et DataLoader, vérifiés sur un batch
from src.dataset import make_loader

images, labels = next(iter(make_loader(df, AIRCRAFT_DIR, "train", True, num_workers=0)))
print("Batch :", images.shape, "| valeurs entre", round(images.min().item(), 2), "et", round(images.max().item(), 2))
print("Labels :", {level: labels[level].shape for level in LEVELS})

Batch : torch.Size([32, 3, 224, 224]) | valeurs entre -2.12 et 2.64
Labels : {'manufacturer': torch.Size([32]), 'family': torch.Size([32]), 'variant': torch.Size([32])}
